# Lab 11 - Ensemble Models: Hazardous Event Classification

This notebook applies **Lab 11 - Ensemble Models** by comparing individual classifiers with voting and boosting-style ensembles.


## Lab 11 concepts used

- Build multiple base learners.
- Compare individual model performance.
- Combine models with soft voting.
- Include a gradient boosting baseline.
- Evaluate with imbalance-aware metrics.

All hazardous-event classifiers use the no-`European_AQI` feature set.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'AML Assignment' and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT


In [ ]:
DATASET_FILENAME = 'global_urban_smog_pm25_hourly.csv'
matches = sorted((PROJECT_ROOT / 'Datasets').glob(f'*/{DATASET_FILENAME}'))
if not matches:
    raise FileNotFoundError(f'Could not find {DATASET_FILENAME} under {PROJECT_ROOT / "Datasets"}')
DATASET_PATH = matches[0]
data = pd.read_csv(DATASET_PATH)
data['Timestamp'] = pd.to_datetime(data['Timestamp'])
data = data.sort_values(['Timestamp', 'City']).reset_index(drop=True)
print(DATASET_PATH)
data.head()


In [ ]:
def add_time_features(df):
    out = df.copy()
    out['Timestamp'] = pd.to_datetime(out['Timestamp'])
    out = out.sort_values(['Timestamp', 'City']).reset_index(drop=True)
    out['hour'] = out['Timestamp'].dt.hour
    out['dayofweek'] = out['Timestamp'].dt.dayofweek
    out['month'] = out['Timestamp'].dt.month
    out['dayofyear'] = out['Timestamp'].dt.dayofyear
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    return out

def chronological_split(df, train_size=0.8):
    split_idx = int(len(df) * train_size)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def latest_rows(df, max_rows):
    if len(df) <= max_rows:
        return df.copy()
    return df.tail(max_rows).copy()

NUMERIC_NO_AQI = [
    'Latitude', 'Longitude', 'PM10_ug_m3', 'PM2_5_ug_m3',
    'Carbon_Monoxide_ug_m3', 'Nitrogen_Dioxide_ug_m3',
    'Ozone_ug_m3', 'Dust_ug_m3', 'UV_Index',
    'hour', 'dayofweek', 'month', 'is_weekend'
]
CATEGORICAL_FEATURES = ['City']


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_preprocess = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_preprocess, NUMERIC_NO_AQI),
        ('cat', categorical_preprocess, CATEGORICAL_FEATURES),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)


In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, precision_recall_curve, f1_score
)


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

model_df = latest_rows(add_time_features(data), 60000)
train_df, test_df = chronological_split(model_df, train_size=0.8)
X_train = train_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_train = train_df['Hazardous_Event']
X_test = test_df[NUMERIC_NO_AQI + CATEGORICAL_FEATURES]
y_test = test_df['Hazardous_Event']

base_models = {
    'Logistic Regression': Pipeline(steps=[('preprocess', preprocessor), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
    'Decision Tree': Pipeline(steps=[('preprocess', preprocessor), ('model', DecisionTreeClassifier(max_depth=8, min_samples_leaf=60, class_weight='balanced', random_state=42))]),
    'Random Forest': Pipeline(steps=[('preprocess', preprocessor), ('model', RandomForestClassifier(n_estimators=80, max_depth=12, min_samples_leaf=30, class_weight='balanced_subsample', n_jobs=-1, random_state=42))]),
    'Hist Gradient Boosting': Pipeline(steps=[('preprocess', preprocessor), ('model', HistGradientBoostingClassifier(max_iter=120, learning_rate=0.06, l2_regularization=0.1, random_state=42))]),
}


In [ ]:
results = []
for name, estimator in base_models.items():
    estimator.fit(X_train, y_train)
    pred = estimator.predict(X_test)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, pred),
        'f1_hazardous': f1_score(y_test, pred),
    })

pd.DataFrame(results).sort_values('f1_hazardous', ascending=False)


In [ ]:
voting_model = VotingClassifier(
    estimators=[
        ('lr', base_models['Logistic Regression']),
        ('dt', base_models['Decision Tree']),
        ('rf', base_models['Random Forest']),
    ],
    voting='soft'
)
voting_model.fit(X_train, y_train)
voting_pred = voting_model.predict(X_test)
print(classification_report(y_test, voting_pred, digits=3))


In [ ]:
ensemble_row = {
    'model': 'Soft Voting Ensemble',
    'accuracy': accuracy_score(y_test, voting_pred),
    'balanced_accuracy': balanced_accuracy_score(y_test, voting_pred),
    'f1_hazardous': f1_score(y_test, voting_pred),
}
comparison_df = pd.concat([pd.DataFrame(results), pd.DataFrame([ensemble_row])], ignore_index=True)
comparison_df.sort_values('f1_hazardous', ascending=False)


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison_df.sort_values('f1_hazardous', ascending=False), x='f1_hazardous', y='model')
plt.xlabel('F1 for hazardous class')
plt.ylabel('Model')
plt.title('Ensemble lab model comparison')
plt.show()


## What was learned from Lab 11

Ensemble methods are useful final-assignment candidates because they combine non-linear modeling strength with measurable model-comparison evidence. The report should still justify any added complexity against simpler baselines.
